In [153]:
import pandas as pd

In [154]:
root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
outputFolder = root + "DataFrames" + "/"

df = pd.read_csv('/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/MainDataFrame_Lineages.csv')

In [155]:
def getonsets(filepath, position, dataset):
    onsets = pd.read_csv(filepath)
    onsets["Source_ID"] = onsets.Label.str.split(":").str.get(1).astype(int)
    onsets["Position"] = position
    onsets["dataset"] = dataset
    onesets = onsets.rename(columns = {"Slice": "Frame_onset"}, inplace = True)
    return onsets

df_onsets_01 = getonsets(
    filepath = root + "/20250627/Position_3_MonastrolWashout/Concatenated/20250627_IM_H2B-GFP_MitoStop_02_onsets.csv",
    position = 3,
    dataset = "20250627"
)

df_onsets_02 = getonsets(
    filepath = root + "/20250627/Position_4_MonastrolWashout/Concatenated/20250627_IM_H2B-GFP_MitoStop_02_onsets.csv",
    position = 4,
    dataset = "20250627"
)

df_onsets_03 = getonsets(
    filepath = root + "/20250704/Position_1_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_onsets.csv",
    position = 1,
    dataset = "20250704"
)

df_onsets_04 = getonsets(
    filepath = root + "/20250704/Position_2_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_onsets.csv",
    position = 2,
    dataset = "20250704"
)

df_onsets_05 = getonsets(
    filepath = root + "/20250704/Position_3_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_onsets.csv",
    position = 3,
    dataset = "20250704"
)

df_onsets_06 = getonsets(
    filepath = root + "/20250704/Position_4_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_onsets.csv",
    position = 4,
    dataset = "20250704"
)

df_onsets_07 = getonsets(
    filepath = root + "/20250704/Position_5_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_onsets.csv",
    position = 5,
    dataset = "20250704"
)

df_onsets_08 = getonsets(
    filepath = root + "/20250704/Position_6_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_onsets.csv",
    position = 6,
    dataset = "20250704"
)

df_onsets_09 = getonsets(
    filepath = root + "/20250704/Position_7_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_onsets.csv",
    position = 7,
    dataset = "20250704"
)

df_onsets = pd.concat(
    [
        df_onsets_01, 
        df_onsets_02,
        df_onsets_03,
        df_onsets_04,
        df_onsets_05,
        df_onsets_06,
        df_onsets_07,
        df_onsets_08,
        df_onsets_09,
        
    ]
)

In [156]:
df["Frame"] = df.Frame.astype(int) + 1 #otherwise the trackmate and mitotic timing frames are not aligned

df = df.merge(df_onsets, how = "outer")

df.head()
    

,Unnamed: 0,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,Position,Dataset,...,Cell_Cycle_hours,Seen_sister,Random_ID,Seen_granny,,Label,X,Y,Frame_onset,dataset
0,0,0,2011,True,2011,444.566899,406.863649,66,3,20250627,...,NaN,NaN,2524,NaN,22,tomeasure:2011:11,171.529,747.022,11,20250627
1,0,0,2012,True,2012,761.937434,648.137100,59,3,20250704,...,NaN,NaN,5249,NaN,52,tomeasure:2012:14,819.528,643.027,14,20250704
2,2,0,2046,True,2012.2046,739.287853,889.272274,461,3,20250704,...,46.900000,NaN,4915,NaN,22,tomeasure:2046:444,742.464,919.794,444,20250704
3,4,1,2081,True,2081,580.740785,683.768758,58,3,20250704,...,NaN,NaN,5249,NaN,46,tomeasure:2081:48,568.449,727.549,48,20250704
4,2,0,2105,True,2011.2105,452.646167,440.838007,527,3,20250627,...,53.783333,True,6515,NaN,76,tomeasure:2105:500,454.096,473.569,500,20250627


In [157]:
df["Frames_in_mitosis"] = (df.Frame.astype(int) - df.Frame_onset.astype(int))
df["Mitotic_duration_mins"] = df.Frames_in_mitosis * 7

In [158]:
def getMitoticDurationMother(x, dataframe = df):
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.dataset == x.dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        
        if len(mother_row) == 1:
            return mother_row.iloc[0].Mitotic_duration_mins
        else:
            return None


def getMetaphaseArrest(x):
    if x > 180:
        return "180+"
    elif x > 120:
        return "120-180"
    elif x > 60:
        return "060-120"
    else:
        return "0-60"

df["Mother_Mitotic_duration_mins"] = df.apply(getMitoticDurationMother, axis = 1)
df["Mother_arrested"] = df.Mother_Mitotic_duration_mins.apply(getMetaphaseArrest)

df.head()

,Unnamed: 0,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,Position,Dataset,...,,Label,X,Y,Frame_onset,dataset,Frames_in_mitosis,Mitotic_duration_mins,Mother_Mitotic_duration_mins,Mother_arrested
0,0,0,2011,True,2011,444.566899,406.863649,66,3,20250627,...,22,tomeasure:2011:11,171.529,747.022,11,20250627,55,385,NaN,0-60
1,0,0,2012,True,2012,761.937434,648.137100,59,3,20250704,...,52,tomeasure:2012:14,819.528,643.027,14,20250704,45,315,NaN,0-60
2,2,0,2046,True,2012.2046,739.287853,889.272274,461,3,20250704,...,22,tomeasure:2046:444,742.464,919.794,444,20250704,17,119,315.0,180+
3,4,1,2081,True,2081,580.740785,683.768758,58,3,20250704,...,46,tomeasure:2081:48,568.449,727.549,48,20250704,10,70,NaN,0-60
4,2,0,2105,True,2011.2105,452.646167,440.838007,527,3,20250627,...,76,tomeasure:2105:500,454.096,473.569,500,20250627,27,189,385.0,180+


In [159]:
df.to_csv(outputFolder + "MainDataFrame_MitoticStopwatch.csv")

print("Finished.")

Finished.


In [160]:
from statistics import mean

def get_lineage_statistics(dataframe = dataframelist, outDir = outputFolder):
    
    dataframes_for_concat = []
        
    for data in dataframe:
        dataset = data
        position = data.loc[0, "Position"]
        max_generation = data.Generation.max()
        No_of_source_ids = data.Source_ID.dropna().shape[0]
        No_of_random_ids = data.Random_ID.dropna().shape[0]
        No_of_sister_ids = data.Sister_ID.dropna().shape[0]
        No_of_mother_ids = data.Mother_ID.dropna().shape[0]
        #No_of_aunt_ids = data.Aunt_ID.dropna().shape[0]
        #No_of_cousin_ids = data.Cousin_ID.dropna().shape[0]
        No_of_grandmother_ids = data.Grandmother_ID.dropna().shape[0]
        
        list_of_tracks = data.Track_ID.unique()
        list_of_trackSubData = []
        for track in list_of_tracks:
            sub_data = data.loc[data.Track_ID == track]
            list_of_trackSubData.append(sub_data)

        No_of_tracks = data.Track_ID.nunique()
        Average_No_of_splitting_events = data.groupby("Track_ID").describe().count().mean()

        sister_per_split = []
        mother_per_split = []
        grandmother_per_split = []
        #aunt_per_split = []
        #cousins_per_split = []

        for sub_data in list_of_trackSubData: 
            number_of_sister_ids = sub_data.Sister_ID.dropna().shape[0]
            sis_per_total_splits = number_of_sister_ids / sub_data.shape[0]
            sister_per_split.append(sis_per_total_splits)

            number_of_mother_ids = sub_data.Mother_ID.dropna().shape[0]
            mom_per_total_splits = number_of_mother_ids / sub_data.shape[0]
            mother_per_split.append(mom_per_total_splits)

            number_of_gmother_ids = sub_data.Grandmother_ID.dropna().shape[0]
            gmom_per_total_splits = number_of_gmother_ids / sub_data.shape[0]
            grandmother_per_split.append(gmom_per_total_splits)

            #number_of_aunt_ids = sub_data.Aunt_ID.dropna().shape[0]
            #aunt_per_total_splits = number_of_aunt_ids / sub_data.shape[0]
            #aunt_per_split.append(aunt_per_total_splits)

            #number_of_cousin_ids = sub_data.Cousin_ID.dropna().shape[0]
            #cousin_per_total_splits = number_of_cousin_ids / sub_data.shape[0]
            #cousins_per_split.append(cousin_per_total_splits)

        statistics_dict = {"Dataset": dataset, 
                           "Position": position, 
                           "Number_of_tracks": No_of_tracks,
                           "Max_No_of_Generations": max_generation,
                           "Number_of_Source_IDs": No_of_source_ids,
                           "Number_of_Sister_IDs": No_of_sister_ids,
                           #"Number_of_Cousin_IDs": No_of_cousin_ids,
                           "Number_of_Mother_IDs": No_of_mother_ids,
                           #"Number_of_Aunt_IDs": No_of_aunt_ids,
                           "Number_of_Grandmother_IDs": No_of_grandmother_ids,
                           "Number_of_Random_IDs": No_of_random_ids,
                           "Average_No_of_SplittingEvents_per_Track": data.shape[0] / No_of_tracks, 
                           "Average_No_of_Sisters_per_Splitting_event": mean(sister_per_split), 
                           "Average_No_of_Mothers_per_Splitting_event": mean(mother_per_split), 
                           "Average_No_of_Grandmothers_per_Splitting_event": mean(grandmother_per_split), 
                           #"Average_No_of_Aunts_per_Splitting_event": mean(aunt_per_split), 
                           #"Average_No_of_Cousin_per_Splitting_event": mean(cousins_per_split)
                          }
        df = pd.DataFrame(statistics_dict, index = [0])
        dataframes_for_concat.append(df)
    
    final_df = pd.concat(dataframes_for_concat)
    final_df.to_csv(outDir + "MetaStatistics_TrackingLineages.csv")
    return final_df

statistics_df = get_lineage_statistics()
statistics_df

NameError: name 'dataframelist' is not defined